# PD optim — 현재 파이프라인 (A10, 단일 idx)

`main_ppmi_pd.py`(최신)를 그대로 호출. **override 없음** — 전부 기본값:
- **SC 정규화 = max** (w/max, 안정 regime, 노드당 입력~1.3)
- **2mm SC** (`FC_AAL_ComBat_all_163.mat`, GROUP_FILTER=PD → 242명)
- activity=1.0, freeze=False, rww_ 배선
- **PART3_REMAT_SCAN=1** (A10 part3 OOM 방지 필수)

PBS가 idx 0~2 돌리는 동안, 여기 A10서 **idx 4** 실행. 다른 subject는 `PD_IDX` 변경(0~241).

⚠ A10에 다른 GPU 작업(예: idx0 검증 run) 있으면 메모리/속도 경쟁 — `nvidia-smi`로 확인.

In [1]:
# ===== JAX/A10 환경 (main_ppmi_pd import 前에 설정) =====
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # A10 전체선점 방지
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ["PART3_REMAT_SCAN"] = "1"   # A10: part3 gradient checkpointing (OOM 방지)

import sys
import main_ppmi_pd as M

# ===== 실행할 PD subject =====
PD_IDX = 4   # 0~241 (PBS가 0~2 돌리는 동안 A10서 idx4)

sys.argv = ["main_ppmi_pd", "--subject-idx", str(PD_IDX)]
M.main()

backend : gpu
devices : [CudaDevice(id=0)]
jax_enable_x64 : False
  EI Tuning Pipeline — AALv3 TR_2.5_PD (163 nodes) (subject_idx=4)
Dataset: AALv3 TR_2.5_PD  (subject idx=4/242, sub_num=100878)
  dropped -> []  → 163 - 0 = 163 nodes
  n_nodes=163  cortex=112  subcortex=51
  SC     -> /scratch/home/wog3597/optim/output_ppmi_pd/100878/inputs/weight.csv   (max=28848)
  length -> /scratch/home/wog3597/optim/output_ppmi_pd/100878/inputs/tract_length.csv  (max=249.0mm)
  FC     -> /scratch/home/wog3597/optim/output_ppmi_pd/100878/inputs/FC.csv   (range=[-0.48,0.89])
  labels -> /scratch/home/wog3597/optim/output_ppmi_pd/100878/inputs/region_labels.txt  (src=data/AALv3/AAL163_labels.txt)
  out_dir-> /scratch/home/wog3597/optim/output_ppmi_pd/100878
  DBS targets (0-based) -> {'STN_L': 161, 'STN_R': 162, 'GP_L': 76, 'GP_R': 77}
[FIG] 출력 폴더: /scratch/home/wog3597/optim/output_ppmi_pd/100878/figures
[cfg] freeze_c_ei_after_fic=False
[cfg] fc_eval_n_seeds=1
[cfg] sc_norm=log1pm
[cfg] use_delay=T

/scratch/home/wog3597/optim/main_ppmi_pd.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _plt._main_original_show(*args, **kwargs)


[MODEL] Running warmup simulation...
[MODEL] Warmup done — E mean=0.4731  I mean=0.1720
[MODEL] Graph type: DenseDelayGraph
StateBundle(stage='warmup', c_ei_frozen=False, init_dyn_shape=(2, 163))

[1] Running FIC...
Running computations for fic_v_pdaal163_tr25_s4_delay3_N163_scd9877b0546_fc7329897119_se0.25_eta0p5_steps2000_dur2500_skip0_bestbundlev4_fpad6855956ff8
[FIC] target S_e=0.25  eta=0.5  max_steps=2000
  step=2s  skip=0 TR  use=1 TR
  step   25/2000  mean_S_e=0.2706  rE=5.412 Hz  se_err=0.0206  (17.0s)
  step   50/2000  mean_S_e=0.2491  rE=4.982 Hz  se_err=0.0009  (28.3s)
  step   75/2000  mean_S_e=0.2473  rE=4.946 Hz  se_err=0.0027  (39.3s)
  step  100/2000  mean_S_e=0.2434  rE=4.867 Hz  se_err=0.0066  (50.6s)
  step  125/2000  mean_S_e=0.2523  rE=5.047 Hz  se_err=0.0023  (61.8s)
  step  150/2000  mean_S_e=0.2588  rE=5.175 Hz  se_err=0.0088  (72.9s)
  step  175/2000  mean_S_e=0.2522  rE=5.044 Hz  se_err=0.0022  (84.2s)
  step  200/2000  mean_S_e=0.2468  rE=4.935 Hz  se_err=0.

/scratch/home/wog3597/optim/main_ppmi_pd.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _plt._main_original_show(*args, **kwargs)


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100878/figures/004_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100878/figures/005_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100878/figures/006_Part_1___FC_matrices.png
[FIC] c_ei_frozen=False  mean c_ei=1.1532
StateBundle(stage='fic', c_ei_frozen=False, init_dyn_shape=(2, 163))

[2] Running EIB...
Running computations for eib_v_pdaal163_tr25_s4_delay3_N163_scd9877b0546_fc7329897119_win240_etaF0p1_etaE0p005_steps10000_phk8_frozen0_fpa62abd55e646
[EIB] 1단계 탐색: 10000스텝 × 1 TR  window=240 TR  c_ei_frozen=False
      Step   Win-corr   Win-RMSE  Best-win-corr    Elapsed        ETA
----------------------------------------------------------------------
      50/10000       0.0145      0.2470          0.0179         31s  1h 44m 56s
     100/10000       0.0219      0.2457          0.0258         57s   1h 35m 4s
     150/10000       0.0225      0.2444          0.0265      1m 23s  1h 

...

  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100878/figures/020_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100878/figures/021_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
[EIB] Block corr post  full=0.7092  ctx-ctx=0.7194  cross=0.7029  sub-sub=0.6840
[EIB] stage=eib  c_ei_frozen=False
StateBundle(stage='eib', c_ei_frozen=False, init_dyn_shape=(2, 163))

[3] Running Gradient Optimization...
Running computations for grad_v_pdaal163_tr25_s4_delay3_N163_scd9877b0546_fc7329897119_TR240_SKIP48_STEPS250_LR0p0001_a0p8_g0p2_rb1_act1p0_bc0p4-0p4-0p2_frozen0_optpersist_fpb84363cef2eb


/scratch/home/wog3597/optim/main_ppmi_pd.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _plt._main_original_show(*args, **kwargs)


[Part3] Initial loss: 0.294550  |  Full Corr: 0.7097
  t1_opt=600000ms (600.0s)  max_steps=250  chunk=10  lr=0.0001  fc_skip_tr=48
[GRAD] Starting full-matrix optimization  (total 250 steps, chunk=10)
      Step         Loss   FullCorr     BestLoss   Step/s    Elapsed        ETA
----------------------------------------------------------------------------------
  [REMAT] PART3_REMAT_SCAN=1 → scan checkpointing ON (AD tape 메모리↓, ~1.5-2x 느림, 결과 동일)


E0728 15:26:52.473498  262481 gpu_hlo_schedule.cc:817] The byte size of input/output arguments (34049167576) exceeds the base limit (18979068313). This indicates an error in the calculation!
W0728 15:26:52.487862  262481 hlo_rematerialization.cc:3198] Can't reduce memory use below 31.35GiB (33657591952 bytes) by rematerialization; only reduced to 32.28GiB (34656456224 bytes), down from 32.28GiB (34656456224 bytes) originally
E0728 15:26:57.623028  262481 pjrt_stream_executor_client.cc:3314] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Failed to allocate request for 30.60GiB (32860800000B) on device ordinal 0


XlaRuntimeError: RESOURCE_EXHAUSTED: Failed to allocate request for 30.60GiB (32860800000B) on device ordinal 0

In [2]:
# ===== JAX/A10 환경 (main_ppmi_pd import 前에 설정) =====
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # A10 전체선점 방지
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ["PART3_REMAT_SCAN"] = "1"   # A10: part3 gradient checkpointing (OOM 방지)

import sys
import main_ppmi_pd as M

# ===== 실행할 PD subject =====
PD_IDX = 5   # 0~241 (PBS가 0~2 돌리는 동안 A10서 idx4)

sys.argv = ["main_ppmi_pd", "--subject-idx", str(PD_IDX)]
M.main()

  EI Tuning Pipeline — AALv3 TR_2.5_PD (163 nodes) (subject_idx=5)
Dataset: AALv3 TR_2.5_PD  (subject idx=5/242, sub_num=100889)
  dropped -> []  → 163 - 0 = 163 nodes
  n_nodes=163  cortex=112  subcortex=51
  SC     -> /scratch/home/wog3597/optim/output_ppmi_pd/100889/inputs/weight.csv   (max=22755)
  length -> /scratch/home/wog3597/optim/output_ppmi_pd/100889/inputs/tract_length.csv  (max=249.0mm)
  FC     -> /scratch/home/wog3597/optim/output_ppmi_pd/100889/inputs/FC.csv   (range=[-0.53,0.90])
  labels -> /scratch/home/wog3597/optim/output_ppmi_pd/100889/inputs/region_labels.txt  (src=data/AALv3/AAL163_labels.txt)
  out_dir-> /scratch/home/wog3597/optim/output_ppmi_pd/100889
  DBS targets (0-based) -> {'STN_L': 161, 'STN_R': 162, 'GP_L': 76, 'GP_R': 77}
[FIG] 출력 폴더: /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures
[cfg] freeze_c_ei_after_fic=False
[cfg] fc_eval_n_seeds=1
[cfg] sc_norm=log1pm
[cfg] use_delay=True  tract_conduction_speed=3.0 mm/ms
  Config
  region_txt       

/scratch/home/wog3597/optim/part2_eib.py:639: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/scratch/home/wog3597/optim/part2_eib.py:639: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/046_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/047_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/048_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/049_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/050_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/051_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/052_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/053_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/054_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/055_Part_2___EIB_Convergence__Po

/scratch/home/wog3597/optim/main_ppmi_pd.py:83: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  fig.savefig(str(out), dpi=150, bbox_inches="tight")
/scratch/home/wog3597/optim/main_ppmi_pd.py:83: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  fig.savefig(str(out), dpi=150, bbox_inches="tight")


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/071_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/072_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/073_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/074_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/075_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/076_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/077_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100889/figures/078_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
[EIB] Block corr post  full=0.7623  ctx-ctx=0.7685  cross=0.7017  sub-sub=0.8406
[EIB] stage=eib  c_ei_frozen=False
StateBundle(stage='eib', c_ei_frozen=False, init_d

/scratch/home/wog3597/optim/main_ppmi_pd.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _plt._main_original_show(*args, **kwargs)


[Part3] Initial loss: 0.233095  |  Full Corr: 0.7642
  t1_opt=600000ms (600.0s)  max_steps=250  chunk=10  lr=0.0001  fc_skip_tr=48
[GRAD] Starting full-matrix optimization  (total 250 steps, chunk=10)
      Step         Loss   FullCorr     BestLoss   Step/s    Elapsed        ETA
----------------------------------------------------------------------------------
  [REMAT] PART3_REMAT_SCAN=1 → scan checkpointing ON (AD tape 메모리↓, ~1.5-2x 느림, 결과 동일)


E0728 18:59:57.209103  262481 gpu_hlo_schedule.cc:817] The byte size of input/output arguments (34049167576) exceeds the base limit (18979068313). This indicates an error in the calculation!
W0728 18:59:57.222396  262481 hlo_rematerialization.cc:3198] Can't reduce memory use below 31.35GiB (33657591952 bytes) by rematerialization; only reduced to 32.28GiB (34656456224 bytes), down from 32.28GiB (34656456224 bytes) originally
E0728 19:00:02.218157  262481 pjrt_stream_executor_client.cc:3314] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Failed to allocate request for 30.60GiB (32860800000B) on device ordinal 0


XlaRuntimeError: RESOURCE_EXHAUSTED: Failed to allocate request for 30.60GiB (32860800000B) on device ordinal 0

In [3]:
# ===== JAX/A10 환경 (main_ppmi_pd import 前에 설정) =====
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # A10 전체선점 방지
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ["PART3_REMAT_SCAN"] = "1"   # A10: part3 gradient checkpointing (OOM 방지)

import sys
import main_ppmi_pd as M

# ===== 실행할 PD subject =====
PD_IDX = 6   # 0~241 (PBS가 0~2 돌리는 동안 A10서 idx4)

sys.argv = ["main_ppmi_pd", "--subject-idx", str(PD_IDX)]
M.main()

  EI Tuning Pipeline — AALv3 TR_2.5_PD (163 nodes) (subject_idx=6)
Dataset: AALv3 TR_2.5_PD  (subject idx=6/242, sub_num=100905)
  dropped -> []  → 163 - 0 = 163 nodes
  n_nodes=163  cortex=112  subcortex=51
  SC     -> /scratch/home/wog3597/optim/output_ppmi_pd/100905/inputs/weight.csv   (max=22591)
  length -> /scratch/home/wog3597/optim/output_ppmi_pd/100905/inputs/tract_length.csv  (max=249.0mm)
  FC     -> /scratch/home/wog3597/optim/output_ppmi_pd/100905/inputs/FC.csv   (range=[-0.51,1.00])
  labels -> /scratch/home/wog3597/optim/output_ppmi_pd/100905/inputs/region_labels.txt  (src=data/AALv3/AAL163_labels.txt)
  out_dir-> /scratch/home/wog3597/optim/output_ppmi_pd/100905
  DBS targets (0-based) -> {'STN_L': 161, 'STN_R': 162, 'GP_L': 76, 'GP_R': 77}
[FIG] 출력 폴더: /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures
[cfg] freeze_c_ei_after_fic=False
[cfg] fc_eval_n_seeds=1
[cfg] sc_norm=log1pm
  Config
  region_txt                                   = /scratch/home/wog3597/opt

/scratch/home/wog3597/optim/part2_eib.py:639: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/scratch/home/wog3597/optim/part2_eib.py:639: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/191_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/192_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/193_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/194_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/195_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/196_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/197_Part_3___Full-matrix_Gradient_Optimization.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/198_Part_3___Full-matrix_Gradient_Optimization___Block-wise_FC_c.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/199_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/

/scratch/home/wog3597/optim/part2_eib.py:647: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, 3, figsize=(12, 4))


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/211_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/212_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/213_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/214_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/215_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/216_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/217_Part_3___Full-matrix_Gradient_Optimization.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/218_Part_3___Full-matrix_Gradient_Optimization___Block-wise_FC_c.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/219_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/

...

/scratch/home/wog3597/optim/main_ppmi_pd.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _plt._main_original_show(*args, **kwargs)


[Part3] Initial loss: 0.225505  |  Full Corr: 0.7977
  t1_opt=600000ms (600.0s)  max_steps=250  chunk=10  lr=0.0001  fc_skip_tr=48
[GRAD] Starting full-matrix optimization  (total 250 steps, chunk=10)
      Step         Loss   FullCorr     BestLoss   Step/s    Elapsed        ETA
----------------------------------------------------------------------------------
  [REMAT] PART3_REMAT_SCAN=1 → scan checkpointing ON (AD tape 메모리↓, ~1.5-2x 느림, 결과 동일)
      10/250         0.220985      0.8013      0.220985      0.03      6m 38s  2h 39m 31s
      20/250         0.217350      0.8039      0.217350      0.03     13m 15s  2h 32m 33s
      30/250         0.214445      0.8056      0.214445      0.03     19m 51s  2h 25m 40s
      40/250         0.212332      0.8066      0.212332      0.03     26m 28s   2h 19m 1s
      50/250         0.210860      0.8072      0.210860      0.03      33m 5s  2h 12m 23s
      60/250         0.209772      0.8079      0.209772      0.03     39m 43s   2h 5m 47s
      70/2

/scratch/home/wog3597/optim/main_ppmi_pd.py:83: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  fig.savefig(str(out), dpi=150, bbox_inches="tight")
/scratch/home/wog3597/optim/main_ppmi_pd.py:83: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  fig.savefig(str(out), dpi=150, bbox_inches="tight")


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/281_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/282_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/283_Part_3___Full-matrix_Gradient_Optimization.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/284_Part_3___Full-matrix_Gradient_Optimization___Block-wise_FC_c.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/285_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/286_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/287_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/288_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100905/figures/289_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/opti

/scratch/home/wog3597/optim/main_ppmi_pd.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _plt._main_original_show(*args, **kwargs)


In [4]:
# ===== JAX/A10 환경 (main_ppmi_pd import 前에 설정) =====
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # A10 전체선점 방지
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ["PART3_REMAT_SCAN"] = "1"   # A10: part3 gradient checkpointing (OOM 방지)

import sys
import main_ppmi_pd as M

# ===== 실행할 PD subject =====
PD_IDX = 0   # 0~241 (PBS가 0~2 돌리는 동안 A10서 idx4)

sys.argv = ["main_ppmi_pd", "--subject-idx", str(PD_IDX)]
M.main()

  EI Tuning Pipeline — AALv3 TR_2.5_PD (163 nodes) (subject_idx=0)
Dataset: AALv3 TR_2.5_PD  (subject idx=0/242, sub_num=100001)
  dropped -> []  → 163 - 0 = 163 nodes
  n_nodes=163  cortex=112  subcortex=51
  SC     -> /scratch/home/wog3597/optim/output_ppmi_pd/100001/inputs/weight.csv   (max=33720)
  length -> /scratch/home/wog3597/optim/output_ppmi_pd/100001/inputs/tract_length.csv  (max=249.0mm)
  FC     -> /scratch/home/wog3597/optim/output_ppmi_pd/100001/inputs/FC.csv   (range=[-0.57,0.99])
  labels -> /scratch/home/wog3597/optim/output_ppmi_pd/100001/inputs/region_labels.txt  (src=data/AALv3/AAL163_labels.txt)
  out_dir-> /scratch/home/wog3597/optim/output_ppmi_pd/100001
  DBS targets (0-based) -> {'STN_L': 161, 'STN_R': 162, 'GP_L': 76, 'GP_R': 77}
[FIG] 출력 폴더: /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures
[cfg] freeze_c_ei_after_fic=False
[cfg] fc_eval_n_seeds=1
[cfg] sc_norm=log1pm
  Config
  region_txt                                   = /scratch/home/wog3597/opt

/scratch/home/wog3597/optim/part2_eib.py:639: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/scratch/home/wog3597/optim/part2_eib.py:639: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/379_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/380_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/381_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/382_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/383_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/384_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/385_Part_3___Full-matrix_Gradient_Optimization.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/386_Part_3___Full-matrix_Gradient_Optimization___Block-wise_FC_c.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/387_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/

/scratch/home/wog3597/optim/main_ppmi_pd.py:83: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  fig.savefig(str(out), dpi=150, bbox_inches="tight")
/scratch/home/wog3597/optim/main_ppmi_pd.py:83: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  fig.savefig(str(out), dpi=150, bbox_inches="tight")


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/439_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/440_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/441_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/442_Part_3___Full-matrix_Gradient_Optimization.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/443_Part_3___Full-matrix_Gradient_Optimization___Block-wise_FC_c.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/444_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/445_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/446_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/447_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] 

/scratch/home/wog3597/optim/main_ppmi_pd.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _plt._main_original_show(*args, **kwargs)


[Part3] Initial loss: 0.370497  |  Full Corr: 0.6134
  t1_opt=600000ms (600.0s)  max_steps=250  chunk=10  lr=0.0001  fc_skip_tr=48
[GRAD] Starting full-matrix optimization  (total 250 steps, chunk=10)
      Step         Loss   FullCorr     BestLoss   Step/s    Elapsed        ETA
----------------------------------------------------------------------------------
  [REMAT] PART3_REMAT_SCAN=1 → scan checkpointing ON (AD tape 메모리↓, ~1.5-2x 느림, 결과 동일)
      10/250         0.364592      0.6198      0.364592      0.02      6m 48s  2h 43m 25s
      20/250         0.359187      0.6255      0.359187      0.02     13m 29s   2h 35m 9s
      30/250         0.354297      0.6306      0.354297      0.02      20m 9s  2h 27m 52s
      40/250         0.350029      0.6350      0.350029      0.02     26m 51s   2h 21m 2s
      50/250         0.346359      0.6388      0.346359      0.02     33m 35s  2h 14m 23s
      60/250         0.343273      0.6421      0.343273      0.03      40m 9s   2h 7m 10s
      70/2

/scratch/home/wog3597/optim/main_ppmi_pd.py:83: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  fig.savefig(str(out), dpi=150, bbox_inches="tight")
/scratch/home/wog3597/optim/main_ppmi_pd.py:83: UserWarning: Glyph 49353 (\N{HANGUL SYLLABLE SAEG}) missing from font(s) DejaVu Sans.
  fig.savefig(str(out), dpi=150, bbox_inches="tight")


  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/500_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/501_Part_2___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/502_Part_2___Block-wise_FC_corr__pre_vs_post-EIB_.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/503_Part_3___Full-matrix_Gradient_Optimization.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/504_Part_3___Full-matrix_Gradient_Optimization___Block-wise_FC_c.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/505_Structural_Weights.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/506_Part_1___FIC_Results.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/507_Part_1___FC_matrices.png
  [fig] /scratch/home/wog3597/optim/output_ppmi_pd/100001/figures/508_Part_2___EIB_Convergence__Post-hoc_Validation_.png
  [fig] 

/scratch/home/wog3597/optim/main_ppmi_pd.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _plt._main_original_show(*args, **kwargs)
